# Comparação de eficiência de CNNs — ISIC 2019

Driver para rodar o estudo no Google Colab.

**Antes de começar:** `Runtime ▸ Change runtime type ▸ GPU`.

Rode as células na ordem. `results/`, `checkpoints/` e `figs/` são gravados
no seu Google Drive, então o progresso sobrevive a quedas de sessão.
O treino é **resumível**: se a sessão cair, re-execute a mesma célula de treino.

## 1. Setup — clona o repositório e instala dependências

Faz um clone limpo a cada execução (o código é pequeno; `results/`,
`checkpoints/` e `figs/` ficam no Drive, então nada se perde).

In [ ]:
import os

REPO_URL = "https://github.com/pedruck/skin-cnn-efficiency.git"
REPO_DIR = "/content/skin-cnn-efficiency"

!rm -rf "{REPO_DIR}" && git clone "{REPO_URL}" "{REPO_DIR}"
assert os.path.isfile(f'{REPO_DIR}/config.yaml'), 'clone falhou — veja o erro acima'
os.chdir(REPO_DIR)
!pip install -q thop kagglehub pyyaml nvidia-ml-py
print('OK — repo em', REPO_DIR)

## 2. Google Drive para as saídas persistentes

As saídas (`results/`, `checkpoints/`, `figs/`) são gravadas direto numa
pasta do seu Drive — os caminhos são escritos no `config.yaml`. Assim o
progresso sobrevive a quedas de sessão.

In [ ]:
import yaml
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = '/content/drive/MyDrive/skin-cnn-efficiency'
for sub in ('results', 'checkpoints', 'figs'):
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths'] = {'results_dir':     f'{DRIVE_ROOT}/results',
                'checkpoints_dir': f'{DRIVE_ROOT}/checkpoints',
                'figures_dir':     f'{DRIVE_ROOT}/figs'}
with open('config.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
print('saidas ->', DRIVE_ROOT)

## 3. Dataset ISIC 2019 (download via kagglehub)

Re-baixa a cada sessão (poucos GB). Se pedir credencial, faça upload do
`kaggle.json` (Kaggle ▸ Account ▸ Create New Token) em `/root/.config/kaggle/`.

In [ ]:
import kagglehub
DATA = kagglehub.dataset_download('salviohexia/isic-2019-skin-lesion-images-for-classification')
print('DATA =', DATA)

## 4. Treino

**Uma arquitetura por célula.** ~25–55 min cada numa T4 (ResNet-50 é a mais lenta).
Se a sessão cair, é só re-executar a célula: retoma da última época salva.

Ajuste as épocas em `config.yaml` (padrão: 20) ou com `--epochs`.

In [ ]:
!python -m src.train --model resnet50 --data "$DATA"

In [ ]:
!python -m src.train --model resnet18 --data "$DATA"

In [ ]:
!python -m src.train --model mobilenet_v2 --data "$DATA"

In [ ]:
!python -m src.train --model mobilenet_v3_large --data "$DATA"

## 5. Benchmark — avaliação no teste + métricas de eficiência

Gera `results/performance.csv`, `efficiency.csv`, `summary.csv`,
`environment.json` e `test_predictions_<modelo>.npz`.

In [ ]:
!python -m src.benchmark --data "$DATA"

## 6. Figuras

In [ ]:
!python -m src.plots --data "$DATA"

import glob, yaml
from IPython.display import Image, display
figs_dir = yaml.safe_load(open('config.yaml'))['paths']['figures_dir']
for f in sorted(glob.glob(f'{figs_dir}/*.png')):
    print(f); display(Image(f))

## 7. Baixar artefatos

Também já estão no seu Drive em `MyDrive/skin-cnn-efficiency/`.

In [ ]:
import os, shutil, yaml
p = yaml.safe_load(open('config.yaml'))['paths']
os.makedirs('/content/artefatos', exist_ok=True)
for key in ('results_dir', 'figures_dir'):
    if os.path.isdir(p[key]):
        shutil.copytree(p[key], f"/content/artefatos/{os.path.basename(p[key])}",
                        dirs_exist_ok=True)
shutil.make_archive('/content/artefatos', 'zip', '/content/artefatos')
from google.colab import files
files.download('/content/artefatos.zip')